In [1]:
# importing relevant libraries
import pandas as pd
import numpy as np
import json
import os
from itertools import chain

# import boto3

from tqdm.notebook import tqdm_notebook
import time

import csv
import re

# display all rows & columns
#pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# show all outputs
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# Disable warnings
import warnings
warnings.filterwarnings("ignore")

**Import data for analysis**

In [2]:
# Import data
attrition_list = 'Hopeliner Copy of Sample 2022 Attrition List.xlsx'
responses = 'TTEC Interviews (Responses).xlsx'

attrition_df = pd.read_excel(attrition_list)
response_df = pd.read_excel(responses)


**Pre-processing of Dataset**

In [3]:
# Strip column names
attrition_df.columns = [x.strip() for x in attrition_df.columns]
response_df.columns = [x.strip() for x in response_df.columns]

# Response df - pre-processing
response_df.rename(columns =  {'Read Call Opening: \nHi (First Name), thank you for making time for this short interview or chat with us. As shared in our (email/ SMS) exchange, my name is (Interviewer Name). I am from e-BI Solutions. We are working with TTEC to find ways to improve their employee experience so we are grateful that you made time for this.':'Read Call Opening',
                              'Confidentiality Notice and Consent:\nPlease note that we will treat our conversation with confidentiality and only disclose information to the extent you wish to share them with TTEC. Our goal is to help them improve the employee experience through an unbiased study and analysis of the data we gather. We will ask for your sign-off on the final notes we will include in the study. Do you agree and consent to this interview/ short chat? (Share option to record video/ audio/ transcript only)\n\n(Wait for response. Proceed with questions as appropriate)': 'Confidentiality Notice and Consent',
                              }, inplace = True)

cols = ['Stated Reason for Leaving TTEC', 'Other considerations for leaving', 'Top 3 Low lights and impact to them',
'Highlights of their stay with TTEC', 'Suggestions on redesigning the experience',
'What would make you consider returning or referring friends and family members to TTEC?',
'Any questions for e-BI or TTEC?']

for col in cols:
    response_df[col] = response_df[col].astype(str)
    
# Add a new column for all reasons for leaving
response_df['All Leaving Reasons'] = response_df['Stated Reason for Leaving TTEC'] + " " + response_df['Other considerations for leaving']

## Pre-processing Descriptive Data

In [4]:
# pip install gensim
# pip install nltk
# pip install matplotlib
# pip install advertools
# Run 'python -m nltk.downloader all' on ubuntu
# pip install typing-extensions --upgrade

In [5]:
#import modules
import nltk
import gensim
import advertools as adv
import matplotlib.pyplot
import os.path
from gensim import corpora
from gensim.models import LsiModel
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from gensim.models.coherencemodel import CoherenceModel
import matplotlib.pyplot as plt

# Latent Dirichlet Allocation (LDA) Topic Analysis Model

**Resources:**
- [Evaluate Topic Models: Latent Dirichlet Allocation (LDA)](https://towardsdatascience.com/evaluate-topic-model-in-python-latent-dirichlet-allocation-lda-7d57484bb5d0)

- [pyLDAvis: Topic Modelling Exploration Tool That Every NLP Data Scientist Should Know](https://neptune.ai/blog/pyldavis-topic-modelling-exploration-tool-that-every-nlp-data-scientist-should-know)

In [6]:
from gensim.utils import simple_preprocess
from nltk.corpus import stopwords

# Stop words
stop_words = stopwords.words('english') + list(adv.stopwords['tagalog'])

# Functions
def sent_to_words(sentences):
    for sentence in sentences:
        # deacc=True removes punctuations
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))
        
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) 
             if word not in stop_words] for doc in texts]

def preprocess_data_indiv(doc_set):
    """
    Input  : docuemnt list
    Purpose: preprocess text (tokenize, removing stopwords, and stemming)
    Output : preprocessed text
    """
    # initialize regex tokenizer
    tokenizer = RegexpTokenizer(r'\w+')
    # create English & Tagalog stop words list
    en_stop = set(stopwords.words('english') + list(adv.stopwords['tagalog']))
    # Create p_stemmer of class PorterStemmer
    p_stemmer = PorterStemmer()
    # list for tokenized documents in loop
    texts = []
    # loop through document list
    for i in doc_set:
        # clean and tokenize document string
        raw = i.lower()
        tokens = tokenizer.tokenize(raw)
        # remove stop words from tokens
        stopped_tokens = [i for i in tokens if not i in en_stop]
        # stem tokens
        #stemmed_tokens = [p_stemmer.stem(i) for i in stopped_tokens]
        # add tokens to list
        texts.append(stopped_tokens)
    return texts


In [7]:
# NLTK Stop words
# import nltk
# nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = stopwords.words('english') + list(adv.stopwords['tagalog'])
stop_words.extend(['from', 'subject', 're', 'edu', 'use','employee','ttec','resign'])
# Define functions for stopwords, bigrams, trigrams and lemmatization
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) if word not in stop_words] for doc in texts]
def make_bigrams(texts):
    return [bigram_mod[doc] for doc in texts]
def make_trigrams(texts):
    return [trigram_mod[bigram_mod[doc]] for doc in texts]
def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV']):
    """https://spacy.io/api/annotation"""
    texts_out = []
    for sent in texts:
        doc = nlp(" ".join(sent)) 
        texts_out.append([token.lemma_ for token in doc if token.pos_ in allowed_postags])
    return texts_out

In [8]:
# Create a copy of the original response dataframe
analysis_df = response_df.copy()

> 1) Remove punctuations, digits and stop words from text

In [9]:
# Remove punctuations & digits
import re
from string import digits
remove_digits = str.maketrans('', '', digits) # remove digits

for col in cols:
    analysis_df[col] = analysis_df[col].apply(lambda x: re.sub(r'[^\w\s]', '', x).translate(remove_digits).strip().lower())

In [10]:
import spacy
# !python -m spacy download en_core_web_lg 
# !python -m spacy download en_core_web_sm

In [11]:
analysis_df.columns

Index(['Timestamp', 'Email Address', 'Read Call Opening',
       'Confidentiality Notice and Consent', 'Name of former employee',
       'Oracle ID of former employee', 'Last day with TTEC (Complete Date)',
       'Stated Reason for Leaving TTEC', 'Other considerations for leaving',
       'Top 3 Low lights and impact to them',
       'Highlights of their stay with TTEC',
       'Suggestions on redesigning the experience',
       'What would make you consider returning or referring friends and family members to TTEC?',
       'Any questions for e-BI or TTEC?',
       'Closing: Thank the person for the time and trust. Show - responses. End call.',
       'How would you rate your overall experience with TTEC on a scale of 1 to 5? 5 being Very Good and 1 being Not Good',
       'All Leaving Reasons'],
      dtype='object')

In [12]:
# Remove stop words
data = analysis_df['Top 3 Low lights and impact to them'].to_list() # field to analyse (MANUAL INPUT)
data_words = list(sent_to_words(data)) # convert words in response to list of words

# Build the bigram and trigram models
bigram = gensim.models.Phrases(data_words, min_count=5, threshold=100) # higher threshold fewer phrases.
trigram = gensim.models.Phrases(bigram[data_words], threshold=100)
# Faster way to get a sentence clubbed as a trigram/bigram
bigram_mod = gensim.models.phrases.Phraser(bigram)
trigram_mod = gensim.models.phrases.Phraser(trigram)

data_words_nostops = remove_stopwords(data_words) # remove stop words

# Form Bigrams
data_words_bigrams = make_bigrams(data_words_nostops)

# Initialize spacy 'en' model, keeping only tagger component (for efficiency)
nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])

# Do lemmatization keeping only noun, adj, vb, adv
data_lemmatized = lemmatization(data_words_bigrams, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV'])

2022-12-21 09:26:56,469 | INFO | phrases.py:497 | learn_vocab | collecting all words and their counts
2022-12-21 09:26:56,470 | INFO | phrases.py:502 | learn_vocab | PROGRESS: at sentence #0, processed 0 words and 0 word types
2022-12-21 09:26:56,473 | INFO | phrases.py:525 | learn_vocab | collected 2021 word types from a corpus of 1702 words (unigram + bigrams) and 107 sentences
2022-12-21 09:26:56,474 | INFO | phrases.py:580 | add_vocab | using 2021 counts as vocab in Phrases<0 vocab, min_count=5, threshold=100, max_vocab_size=40000000>
2022-12-21 09:26:56,475 | INFO | phrases.py:497 | learn_vocab | collecting all words and their counts
2022-12-21 09:26:56,476 | INFO | phrases.py:502 | learn_vocab | PROGRESS: at sentence #0, processed 0 words and 0 word types
2022-12-21 09:26:56,484 | INFO | phrases.py:525 | learn_vocab | collected 2021 word types from a corpus of 1702 words (unigram + bigrams) and 107 sentences
2022-12-21 09:26:56,485 | INFO | phrases.py:580 | add_vocab | using 2021

> 2) Create a corpus

In [13]:
import gensim.corpora as corpora

def prepare_corpus(doc_clean):
    """
    Input  : clean document
    Purpose: create term dictionary of our courpus and Converting list of documents (corpus) into Document Term Matrix
    Output : term dictionary and Document Term Matrix
    """
    # Creating the term dictionary of our courpus, where every unique term is assigned an index. dictionary = corpora.Dictionary(doc_clean)
    dictionary = corpora.Dictionary(doc_clean)
    # Converting list of documents (corpus) into Document Term Matrix using dictionary prepared above.
    doc_term_matrix = [dictionary.doc2bow(doc) for doc in doc_clean]
    # generate LDA model
    return dictionary,doc_term_matrix

In [14]:
import gensim.corpora as corpora
# Create Dictionary
id2word = corpora.Dictionary(data_lemmatized)
# Create Corpus
texts = data_lemmatized
# Term Document Frequency
corpus = [id2word.doc2bow(text) for text in texts]

2022-12-21 09:26:57,404 | INFO | dictionary.py:209 | add_documents | adding document #0 to Dictionary(0 unique tokens: [])
2022-12-21 09:26:57,406 | INFO | dictionary.py:214 | add_documents | built Dictionary(400 unique tokens: ['agent', 'attack', 'belt', 'company', 'employee']...) from 107 documents (total 794 corpus positions)


> 3) LDA Model 

In [15]:
from pprint import pprint

In [16]:
# number of topics
num_topics = 10

# Build LDA model
lda_model = gensim.models.LdaMulticore(corpus=corpus,
                                       id2word=id2word,
                                       num_topics=num_topics,
                                      random_state=3,
                                      chunksize=100,
                                       passes=10,
                                       per_word_topics=True)
# Print the Keyword in the 10 topics
pprint(lda_model.print_topics())
doc_lda = lda_model[corpus]

2022-12-21 09:26:57,415 | INFO | ldamodel.py:557 | init_dir_prior | using symmetric alpha at 0.1
2022-12-21 09:26:57,415 | INFO | ldamodel.py:557 | init_dir_prior | using symmetric eta at 0.1
2022-12-21 09:26:57,416 | INFO | ldamodel.py:481 | __init__ | using serial LDA version on this node
2022-12-21 09:26:57,417 | INFO | ldamulticore.py:238 | update | running online LDA training, 10 topics, 10 passes over the supplied corpus of 107 documents, updating every 700 documents, evaluating every ~107 documents, iterating 50x with a convergence threshold of 0.001000
2022-12-21 09:26:57,427 | INFO | ldamulticore.py:279 | update | training LDA model using 7 processes
2022-12-21 09:26:57,483 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 0, dispatched chunk #0 = documents up to #100/107, outstanding queue size 1
2022-12-21 09:26:57,489 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 0, dispatched chunk #1 = documents up to #107/107, outstanding queue size 2
2022-12-21 09:26:59,81

2022-12-21 09:26:59,896 | INFO | ldamodel.py:1171 | show_topics | topic #6 (0.100): 0.020*"pay" + 0.019*"benefit" + 0.019*"rest" + 0.019*"training" + 0.018*"incentive" + 0.018*"basic" + 0.018*"well" + 0.018*"feel" + 0.018*"allowance" + 0.018*"generous"
2022-12-21 09:26:59,897 | INFO | ldamodel.py:1171 | show_topics | topic #0 (0.100): 0.043*"work" + 0.032*"leave" + 0.025*"hard" + 0.024*"workmate" + 0.018*"account" + 0.017*"stay" + 0.017*"company" + 0.017*"agent" + 0.015*"superior" + 0.009*"lot"
2022-12-21 09:26:59,897 | INFO | ldamodel.py:1171 | show_topics | topic #5 (0.100): 0.035*"salary" + 0.018*"pay" + 0.018*"need" + 0.018*"high" + 0.018*"tl" + 0.018*"point" + 0.018*"fully" + 0.018*"compare" + 0.018*"competitive" + 0.018*"cover"
2022-12-21 09:26:59,898 | INFO | ldamodel.py:1171 | show_topics | topic #2 (0.100): 0.055*"agent" + 0.033*"supervisor" + 0.024*"time" + 0.024*"transfer" + 0.023*"motivate" + 0.023*"instead" + 0.023*"employee" + 0.023*"tend" + 0.012*"can" + 0.012*"need"
202

2022-12-21 09:26:59,967 | INFO | ldamodel.py:1171 | show_topics | topic #3 (0.100): 0.031*"slow" + 0.021*"process" + 0.021*"hard" + 0.011*"wfh" + 0.011*"able" + 0.011*"answer" + 0.011*"replace" + 0.011*"compensation" + 0.011*"customer" + 0.011*"turn"
2022-12-21 09:26:59,968 | INFO | ldamodel.py:1049 | do_mstep | topic diff=0.010526, rho=0.315127
2022-12-21 09:26:59,970 | INFO | ldamodel.py:822 | log_perplexity | -8.347 per-word bound, 325.6 perplexity estimate based on a held-out corpus of 7 documents with 21 words
2022-12-21 09:26:59,970 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 9, dispatched chunk #0 = documents up to #100/107, outstanding queue size 1
2022-12-21 09:26:59,972 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 9, dispatched chunk #1 = documents up to #107/107, outstanding queue size 2
2022-12-21 09:26:59,981 | INFO | ldamodel.py:1171 | show_topics | topic #6 (0.100): 0.019*"pay" + 0.019*"benefit" + 0.019*"rest" + 0.019*"training" + 0.019*"incentive" +

[(0,
  '0.047*"work" + 0.032*"leave" + 0.024*"hard" + 0.024*"workmate" + '
  '0.022*"account" + 0.016*"stay" + 0.016*"company" + 0.016*"agent" + '
  '0.016*"superior" + 0.009*"lot"'),
 (1,
  '0.050*"pay" + 0.029*"trainer" + 0.022*"final" + 0.022*"setup" + '
  '0.015*"first" + 0.015*"even" + 0.015*"include" + 0.015*"time" + 0.015*"say" '
  '+ 0.015*"experience"'),
 (2,
  '0.057*"agent" + 0.035*"supervisor" + 0.024*"time" + 0.024*"transfer" + '
  '0.024*"motivate" + 0.024*"instead" + 0.024*"employee" + 0.024*"tend" + '
  '0.012*"can" + 0.012*"need"'),
 (3,
  '0.031*"slow" + 0.021*"process" + 0.021*"hard" + 0.011*"wfh" + 0.011*"able" '
  '+ 0.011*"answer" + 0.011*"replace" + 0.011*"compensation" + '
  '0.011*"customer" + 0.011*"turn"'),
 (4,
  '0.030*"salary" + 0.020*"employee" + 0.020*"give" + 0.015*"raise" + '
  '0.015*"company" + 0.015*"day" + 0.015*"health" + 0.015*"question" + '
  '0.010*"issue" + 0.010*"tl"'),
 (5,
  '0.035*"salary" + 0.018*"pay" + 0.018*"need" + 0.018*"high" + 0.01

> 4) Select optimal number of topics

In [17]:
# supporting function
def compute_coherence_values(corpus, dictionary, k, a, b):
    
    lda_model = gensim.models.LdaMulticore(corpus=corpus,
                                           id2word=dictionary,
                                           num_topics=k, 
                                           random_state=100,
                                           chunksize=100,
                                           passes=10,
                                           alpha=a,
                                           eta=b)
    
    coherence_model_lda = CoherenceModel(model=lda_model, texts=data_lemmatized, dictionary=id2word, coherence='c_v')
    
    return coherence_model_lda.get_coherence()

In [18]:
# Baseline coherence score
from gensim.models import CoherenceModel
# Compute Coherence Score
coherence_model_lda = CoherenceModel(model=lda_model, texts=data_lemmatized, dictionary=id2word, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('\nBaseline Coherence Score: ', coherence_lda)

2022-12-21 09:27:00,039 | INFO | probability_estimation.py:155 | p_boolean_sliding_window | using ParallelWordOccurrenceAccumulator(processes=7, batch_size=64) to estimate probabilities from sliding windows
2022-12-21 09:27:01,812 | INFO | text_analysis.py:530 | terminate_workers | 7 accumulators retrieved from output queue
2022-12-21 09:27:01,837 | INFO | text_analysis.py:552 | merge_accumulators | accumulated word occurrence stats for 99 virtual documents



Baseline Coherence Score:  0.35226757870608305


In [19]:
# Optimize coherence score
import tqdm
grid = {}
grid['Validation_Set'] = {}
# Topics range
min_topics = 2
max_topics = 11
step_size = 1
topics_range = range(min_topics, max_topics, step_size)

# Alpha parameter
alpha = list(np.arange(0.01, 1, 0.3))
alpha.append('symmetric')
alpha.append('asymmetric')

# Beta parameter
beta = list(np.arange(0.01, 1, 0.3))
beta.append('symmetric')

# Validation sets
num_of_docs = len(corpus)
corpus_sets = [
#     gensim.utils.ClippedCorpus(corpus, int(num_of_docs*0.25)), 
#                gensim.utils.ClippedCorpus(corpus, int(num_of_docs*0.5)), 
               gensim.utils.ClippedCorpus(corpus, int(num_of_docs*0.75)), 
               corpus]

corpus_title = [
    #'25% Corpus','50% Corpus',
    '75% Corpus', '100% Corpus']

model_results = {'Validation_Set': [],
                 'Topics': [],
                 'Alpha': [],
                 'Beta': [],
                 'Coherence': []
                }

In [20]:
# # iterate through validation corpuses
# for i in tqdm_notebook(range(len(corpus_sets))):
#     # iterate through number of topics
#     for k in topics_range:
#         # iterate through alpha values
#         for a in alpha:
#             # iterare through beta values
#             for b in beta:
#                 # get the coherence score for the given parameters
#                 cv = compute_coherence_values(corpus=corpus_sets[i], dictionary=id2word, 
#                                               k=k, a=a, b=b)
#                 # Save the model results
#                 model_results['Validation_Set'].append(corpus_title[i])
#                 model_results['Topics'].append(k)
#                 model_results['Alpha'].append(a)
#                 model_results['Beta'].append(b)
#                 model_results['Coherence'].append(cv)


In [21]:
# # Create coherence dataframe
# coherence_results = (pd.DataFrame(model_results)
#                      .sort_values(by = 'Coherence')
#                     )

In [22]:
# # Plot results - find optimal alpha & beta to find best coherence, then reconfigure alpha & beta
# plot_df = (coherence_results
#              #.query("(Alpha != 'asymmetric') & (Alpha != 'symmetric') ", engine = 'python')
#              #.query("(Validation_Set != '25% Corpus') & (Validation_Set != '50% Corpus')", engine = 'python')
#              .query("(Alpha == 0.61) & (Beta == 0.61)", engine = 'python')
#              .sort_values(by = 'Topics', ascending = False)             
#             )
# plot_df

# # Plot
# plt.plot(plot_df['Topics'].to_list(), plot_df['Coherence'].to_list())
# plt.show()

In [23]:
# # Select optimal alpha & beta
# (coherence_results
#  .query("Topics == 6")
#  .sort_values(by = 'Coherence', ascending = False)
# )

> 5) Train final model with optimized hyperparameters

In [24]:
# Train final model
lda_model = gensim.models.LdaMulticore(corpus=corpus,
                                           id2word=id2word,
                                           num_topics=6, 
                                           random_state=3,
                                           chunksize=100,
                                           passes=10,
                                           alpha=0.9099999999999999,
                                           eta=0.9099999999999999)

2022-12-21 09:27:02,226 | INFO | ldamodel.py:481 | __init__ | using serial LDA version on this node
2022-12-21 09:27:02,226 | INFO | ldamulticore.py:238 | update | running online LDA training, 6 topics, 10 passes over the supplied corpus of 107 documents, updating every 700 documents, evaluating every ~107 documents, iterating 50x with a convergence threshold of 0.001000
2022-12-21 09:27:02,227 | INFO | ldamulticore.py:279 | update | training LDA model using 7 processes
2022-12-21 09:27:02,273 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 0, dispatched chunk #0 = documents up to #100/107, outstanding queue size 1
2022-12-21 09:27:02,280 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 0, dispatched chunk #1 = documents up to #107/107, outstanding queue size 2
2022-12-21 09:27:04,119 | INFO | ldamodel.py:1171 | show_topics | topic #4 (0.910): 0.012*"salary" + 0.011*"none" + 0.008*"employee" + 0.006*"work" + 0.006*"take" + 0.006*"give" + 0.005*"pay" + 0.005*"process" + 0.0

2022-12-21 09:27:04,265 | INFO | ldamodel.py:1171 | show_topics | topic #2 (0.910): 0.018*"none" + 0.013*"agent" + 0.008*"employee" + 0.006*"supervisor" + 0.006*"salary" + 0.006*"motivate" + 0.006*"tend" + 0.006*"instead" + 0.005*"time" + 0.005*"transfer"
2022-12-21 09:27:04,265 | INFO | ldamodel.py:1171 | show_topics | topic #1 (0.910): 0.014*"pay" + 0.011*"none" + 0.010*"trainer" + 0.008*"good" + 0.007*"final" + 0.007*"even" + 0.006*"include" + 0.006*"first" + 0.006*"say" + 0.005*"day"
2022-12-21 09:27:04,266 | INFO | ldamodel.py:1171 | show_topics | topic #4 (0.910): 0.012*"give" + 0.012*"salary" + 0.010*"process" + 0.009*"take" + 0.008*"none" + 0.008*"question" + 0.008*"employee" + 0.008*"day" + 0.007*"health" + 0.007*"training"
2022-12-21 09:27:04,267 | INFO | ldamodel.py:1171 | show_topics | topic #0 (0.910): 0.015*"work" + 0.014*"none" + 0.012*"leave" + 0.009*"hard" + 0.007*"workmate" + 0.006*"company" + 0.006*"shift" + 0.006*"stay" + 0.006*"agent" + 0.005*"enough"
2022-12-21 09

2022-12-21 09:27:04,388 | INFO | ldamodel.py:1049 | do_mstep | topic diff=0.015686, rho=0.315127
2022-12-21 09:27:04,390 | INFO | ldamodel.py:822 | log_perplexity | -6.866 per-word bound, 116.6 perplexity estimate based on a held-out corpus of 7 documents with 21 words
2022-12-21 09:27:04,391 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 9, dispatched chunk #0 = documents up to #100/107, outstanding queue size 1
2022-12-21 09:27:04,393 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 9, dispatched chunk #1 = documents up to #107/107, outstanding queue size 2
2022-12-21 09:27:04,413 | INFO | ldamodel.py:1171 | show_topics | topic #1 (0.910): 0.016*"pay" + 0.010*"trainer" + 0.010*"good" + 0.010*"none" + 0.008*"final" + 0.008*"even" + 0.006*"feel" + 0.006*"include" + 0.006*"experience" + 0.006*"say"
2022-12-21 09:27:04,414 | INFO | ldamodel.py:1171 | show_topics | topic #3 (0.910): 0.026*"none" + 0.008*"slow" + 0.007*"compensation" + 0.006*"wfh" + 0.005*"stay" + 0.005*"repl

> 4) Visualizing topics

For (1), you can manually select each topic to view its top most frequent and/or “relevant” terms, using different values of the λ parameter. This can help when you’re trying to assign a human interpretable name or “meaning” to each topic.

For (2), exploring the Intertopic Distance Plot can help you learn about how topics relate to each other, including potential higher-level structure between groups of topics.

**Interpretation:**


*Bar chart*
- Each bubble represents a topic. The larger the bubble, the higher percentage of the number of responses in the corpus is about that topic.
- Blue bars represent the overall frequency of each word in the corpus. If no topic is selected, the blue bars of the most frequently used words will be displayed.
- Red bars give the estimated number of times a given term was generated by a given topic. The word with the longest red bar is the word that is used the most by the responses belonging to that topic.
- <u>*Lambda*</u> -- Decreasing the lambda parameter, increases the weight of the ratio of the frequency of word given the topic / Overall frequency of the word in the documents. Important words for the given topic moves upward.
    - Lambda = 1, top words by frequency
    - Lambda = 0, top words by relevance to topic

*Bubble plot*
- The further the bubbles are away from each other, the more different they are. 
- The larger the bubble, the more frequent is the topic in the documents.
- Distance between the topics is an approximation of semantic relationship between the topics.
- The topic which shares common words will be overlapping (closer in distance) in comparison to the non-overlapping topic.
- A good topic model will have big and non-overlapping bubbles scattered throughout the chart. As we can see from the graph, the bubbles are clustered within one place.
- A topic model with a low number of topics will have big non-overlapping bubbles, scattered throughout the chart whereas, the topic model with a high number of topics, will have many overlapping small size bubbles, clustered in the chart.


In [25]:
import pyLDAvis.gensim
import pickle 
import pyLDAvis

In [26]:
# Visualize the topics
pyLDAvis.enable_notebook()
LDAvis_prepared = pyLDAvis.gensim.prepare(lda_model, corpus, id2word)
LDAvis_prepared = pyLDAvis.gensim.prepare(lda_model, corpus, id2word)
LDAvis_prepared

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
4      0.008953 -0.030348       1        1  23.480692
5     -0.031492 -0.000298       2        1  19.804879
1      0.006902  0.006890       3        1  16.965750
0      0.014526  0.014199       4        1  16.838390
2     -0.001965  0.004140       5        1  12.440867
3      0.003075  0.005417       6        1  10.469422, topic_info=         Term      Freq     Total Category  logprob  loglift
32     salary  7.000000  7.000000  Default  30.0000  30.0000
176      none  9.000000  9.000000  Default  29.0000  29.0000
68       work  4.000000  4.000000  Default  28.0000  28.0000
0       agent  4.000000  4.000000  Default  27.0000  27.0000
129     leave  3.000000  3.000000  Default  26.0000  26.0000
..        ...       ...       ...      ...      ...      ...
165      take  0.409210  3.478678   Topic6  -5.3139   0.1165
29       need  0.369959  3.293938   Topic6  -5.4147   0.0703
332      stat  0.284807  1.931104   Topic6  -5.6763   0.3427
323  negative  0.295801  2.217612   Topic6  -5.6384   0.2422
129     leave  0.322809  3.242273   Topic6  -5.5511  -0.0503

[268 rows x 6 columns], token_table=      Topic      Freq      Term
term                           
310       1  0.418281      able
34        2  0.300012   account
0         1  0.238730     agent
0         2  0.238730     agent
0         4  0.238730     agent
...     ...       ...       ...
21        3  0.398759      word
68        1  0.212574      work
68        3  0.212574      work
68        4  0.637722      work
134       4  0.447874  workmate

[182 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[5, 6, 2, 1, 3, 4])

# Post-modeling Exploratory Data Analysis

*Assign respondents to their topic clusters*

In [27]:
# Topic distribution for the whole document. Each element in the list is a pair of a topic’s id, and the probability that was assigned to it.
topic_dist = list(lda_model.get_document_topics(corpus))


# Classify responses to topics
cluster = []
probability = []

for i in range(0, len(topic_dist)):
    clust = [x for x,y in topic_dist[i]]
    prob = [y for x,y in topic_dist[i]]
    get_index = prob.index(max(prob))
    max_val = topic_dist[i][get_index]
    cluster.append(max_val[0])
    probability.append(max_val[1])
    
# Add classification to df
analysis_df['Topic'] = cluster
analysis_df['Topic_Probability'] = probability

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [28]:
# Map topics to Oracle ID
id_topic_mapper = dict(zip(analysis_df['Oracle ID of former employee'], analysis_df['Topic']))
id_prob_mapper = dict(zip(analysis_df['Oracle ID of former employee'], analysis_df['Topic_Probability']))
id_rate_mapper = dict(zip(analysis_df['Oracle ID of former employee'], analysis_df['How would you rate your overall experience with TTEC on a scale of 1 to 5? 5 being Very Good and 1 being Not Good']))

# For those with missing Oracle ID, map to names
name_mapper = (analysis_df[~analysis_df.applymap(np.isreal)['Oracle ID of former employee']]
               .assign(Name = lambda x: x['Name of former employee'].apply(lambda x: x.upper()))
               [['Name','Topic']]
                
                     .set_index("Name").to_dict()['Topic']
)

name_mapper2 = (analysis_df[~analysis_df.applymap(np.isreal)['Oracle ID of former employee']]
               .assign(Name = lambda x: x['Name of former employee'].apply(lambda x: x.upper()))
               [['Name','Topic_Probability']]
                
                     .set_index("Name").to_dict()['Topic_Probability']
)

rating_mapper = (analysis_df[~analysis_df.applymap(np.isreal)['Oracle ID of former employee']]
               .assign(Name = lambda x: x['Name of former employee'].apply(lambda x: x.upper()))
               [['Name','How would you rate your overall experience with TTEC on a scale of 1 to 5? 5 being Very Good and 1 being Not Good']]
                
                     .set_index("Name").to_dict()['How would you rate your overall experience with TTEC on a scale of 1 to 5? 5 being Very Good and 1 being Not Good']
)

# Filter attrition df to contain respondents only
respondents_demog_df = (attrition_df
                         .assign(Topic1 = lambda x: x['EE Oracle Id'].map(id_topic_mapper),
                                 Prob1 = lambda x: x['EE Oracle Id'].map(id_prob_mapper),
                                 Rate1 = lambda x: x['EE Oracle Id'].map(id_rate_mapper),
                                 Name = lambda x: (x['First Name'] + " " + x['Last Name']).apply(lambda x: x.upper()),
                                 Topic = lambda x: np.where(x['Topic1'].isna(), x['Name'].map(name_mapper), x['Topic1']),
                                 Topic_Probability = lambda x: np.where(x['Prob1'].isna(), x['Name'].map(name_mapper2), x['Prob1']),
                                Rating = lambda x: np.where(x['Rate1'].isna(), x['Name'].map(rating_mapper), x['Rate1']),

                         )
                         .drop(['Topic1','Prob1','Rate1'], axis = 1)
                         .query("Topic.notna()", engine = 'python')
                         .sort_values(by = 'Topic')
                        )

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


> 1) How many respondents belong to each topic cluster & what is the average probability score?

In [29]:
# "Number of Employees per Cluster & Average Probability Score")
gen_stats = (analysis_df
             .groupby(['Topic']).agg({'Oracle ID of former employee':'nunique', 'Topic_Probability':'describe'})
             .reset_index()
            )

gen_stats

print("Number of Employees per Cluster & Average Probability Score")


/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Topic Oracle ID of former employee Topic_Probability                      \
        Oracle ID of former employee             count      mean       std   
0     0                           18              18.0  0.406362  0.234584   
1     1                           10              10.0  0.463371  0.238454   
2     2                            7               7.0  0.460676  0.190273   
3     3                           39              39.0  0.242516  0.122621   
4     4                           13              13.0  0.567179  0.194124   
5     5                           20              20.0  0.494275  0.174780   

                                                     
        min       25%       50%       75%       max  
0  0.166667  0.166667  0.360421  0.605164  0.813368  
1  0.224988  0.260636  0.429479  0.531739  0.871821  
2  0.268687  0.342324  0.427150  0.507578  0.829090  
3  0.172714  0.201577  0.201647  0.201750  0.726168  
4  0.204675  0.420357  0.573553  0.676011  0.884534  
5  0.248996  0.350404  0.511455  0.615312  0.827422

Number of Employees per Cluster & Average Probability Score


> 2) Tenure, Topic Probability, and Employee Count by By Attrition Stage

In [30]:
(respondents_demog_df
 .groupby(['Topic','Stage Attrition']).agg({'EE Oracle Id':'nunique','Tenure In Months':['mean', 'std','median'],'Topic_Probability':['mean', 'std','median']})
 .unstack('Stage Attrition')
)

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


EE Oracle Id            Tenure In Months             \
                       nunique                        mean              
Stage Attrition Pre Production Production   Pre Production Production   
Topic                                                                   
0.0                        1.0       17.0         4.000000  27.823529   
1.0                        NaN       10.0              NaN  21.500000   
2.0                        1.0        6.0         3.000000  20.666667   
3.0                        7.0       32.0         5.428571  23.468750   
4.0                        2.0       11.0        11.000000  24.909091   
5.0                        3.0       17.0         2.666667  32.117647   

                                                                     \
                           std                    median              
Stage Attrition Pre Production Production Pre Production Production   
Topic                                                                 
0.0                        NaN  22.622542            4.0       22.0   
1.0                        NaN  18.674106            NaN       18.0   
2.0                        NaN  13.291601            3.0       17.5   
3.0                  10.845232  19.012703            1.0       18.5   
4.0                  14.142136  36.607252           11.0       12.0   
5.0                   2.081666  16.499554            2.0       26.0   

                Topic_Probability                                       \
                             mean                       std              
Stage Attrition    Pre Production Production Pre Production Production   
Topic                                                                    
0.0                      0.698691   0.389166            NaN   0.229812   
1.0                           NaN   0.463371            NaN   0.238454   
2.0                      0.268687   0.492674            NaN   0.186666   
3.0                      0.230280   0.245193       0.089232   0.129803   
4.0                      0.448463   0.588763       0.132616   0.200328   
5.0                      0.536821   0.486767       0.134028   0.183387   

                                           
                        median             
Stage Attrition Pre Production Production  
Topic                                      
0.0                   0.698691   0.314152  
1.0                        NaN   0.429479  
2.0                   0.268687   0.460130  
3.0                   0.201624   0.201647  
4.0                   0.448463   0.600662  
5.0                   0.535909   0.487000

> 3) Tenure, Topic Probability, and Employee Count by By Job Name

In [31]:
(respondents_demog_df
 .groupby(['Topic','Job Name','Stage Attrition']).agg({'EE Oracle Id':'nunique','Tenure In Months':['mean', 'std','median']})
 .unstack('Job Name')
)

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


EE Oracle Id                                        \
                           nunique                                         
Job Name                     CSR I CSR II CSR III Chat Sales Associate I   
Topic Stage Attrition                                                      
0.0   Pre Production           1.0    NaN     NaN                    NaN   
      Production               2.0   11.0     2.0                    NaN   
1.0   Production               2.0    4.0     2.0                    1.0   
2.0   Pre Production           1.0    NaN     NaN                    NaN   
      Production               NaN    6.0     NaN                    NaN   
3.0   Pre Production           3.0    4.0     NaN                    NaN   
      Production               3.0   21.0     6.0                    NaN   
4.0   Pre Production           NaN    2.0     NaN                    NaN   
      Production               2.0    8.0     1.0                    NaN   
5.0   Pre Production           1.0    2.0     NaN                    NaN   
      Production               4.0   11.0     2.0                    NaN   

                                          Tenure In Months             \
                                                      mean              
Job Name              ISR I TSR I TSR III            CSR I     CSR II   
Topic Stage Attrition                                                   
0.0   Pre Production    NaN   NaN     NaN         4.000000        NaN   
      Production        1.0   NaN     1.0        14.500000  29.636364   
1.0   Production        NaN   NaN     1.0        21.500000  23.500000   
2.0   Pre Production    NaN   NaN     NaN         3.000000        NaN   
      Production        NaN   NaN     NaN              NaN  20.666667   
3.0   Pre Production    NaN   NaN     NaN         1.666667   8.250000   
      Production        NaN   1.0     1.0        37.666667  18.857143   
4.0   Pre Production    NaN   NaN     NaN              NaN  11.000000   
      Production        NaN   NaN     NaN        38.000000  22.125000   
5.0   Pre Production    NaN   NaN     NaN         5.000000   1.500000   
      Production        NaN   NaN     NaN        36.250000  29.545455   

                                                                             \
                                                                              
Job Name                 CSR III Chat Sales Associate I ISR I TSR I TSR III   
Topic Stage Attrition                                                         
0.0   Pre Production         NaN                    NaN   NaN   NaN     NaN   
      Production       15.500000                    NaN  59.0   NaN    28.0   
1.0   Production       10.500000                   40.0   NaN   NaN    17.0   
2.0   Pre Production         NaN                    NaN   NaN   NaN     NaN   
      Production             NaN                    NaN   NaN   NaN     NaN   
3.0   Pre Production         NaN                    NaN   NaN   NaN     NaN   
      Production       28.833333                    NaN   NaN  56.0    13.0   
4.0   Pre Production         NaN                    NaN   NaN   NaN     NaN   
      Production       21.000000                    NaN   NaN   NaN     NaN   
5.0   Pre Production         NaN                    NaN   NaN   NaN     NaN   
      Production       38.000000                    NaN   NaN   NaN     NaN   

                                                                               \
                             std                                                
Job Name                   CSR I     CSR II    CSR III Chat Sales Associate I   
Topic Stage Attrition                                                           
0.0   Pre Production         NaN        NaN        NaN                    NaN   
      Production        3.535534  25.511851   0.707107                    NaN   
1.0   Production       27.577164  23.187640  12.020815                    NaN   
2.0   Pre Production         NaN

> 4) Tenure, Topic Probability, and Employee Count by By Job Name

In [32]:
(respondents_demog_df
 .groupby(['Topic','Stage Attrition']).agg({'Rating': ['mean', 'std','median']})
 .round(1)
)

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Rating            
                        mean  std median
Topic Stage Attrition                   
0.0   Pre Production     NaN  NaN    NaN
      Production         4.3  0.6    4.0
1.0   Production         4.1  1.1    4.0
2.0   Pre Production     4.0  NaN    4.0
      Production         3.5  1.7    4.0
3.0   Pre Production     4.7  0.5    5.0
      Production         4.4  0.5    4.0
4.0   Pre Production     3.0  NaN    3.0
      Production         4.1  0.8    4.0
5.0   Pre Production     3.7  1.2    3.0
      Production         3.9  0.7    4.0